In [17]:
# 1. THƯ VIỆN XỬ LÝ VÀ SINH DỮ LIỆU
import numpy as np         
import pandas as pd       
import os                  # Dùng để tạo và quản lý cấu trúc thư mục (data/, models/) 
import random              # Hỗ trợ tạo các biến danh mục ngẫu nhiên
np.random.seed(42)
n_samples = 50000 
print("⏳Bắt đầu tạo dữ liệu...")

⏳Bắt đầu tạo dữ liệu...


In [18]:
# 2. THIẾT KẾ CẤU TRÚC DỮ LIỆU - DATA STRUCTURE DESIGN
# Dựa trên lĩnh vực thực tế (Mobile App Customer Churn Prediction)

print("\n" + "="*60)
print("📊 THIẾT KẾ CẤU TRÚC DỮ LIỆU KHÁCH HÀNG CHURN")
print("="*60)

data_structure = {
    'customer_id': 'ID khách hàng duy nhất',
    'age': 'Tuổi khách hàng (18-70)',
    'account_tenure_months': 'Thời gian sử dụng app (tháng)',
    'monthly_active_days': 'Số ngày hoạt động/tháng',
    'avg_daily_usage_minutes': 'Trung bình thời gian sử dụng/ngày',
    'num_sessions_per_week': 'Số phiên hoạt động/tuần',
    'support_tickets_30d': 'Số ticket hỗ trợ 30 ngày gần nhất',
    'payment_delay_days': 'Số ngày trễ thanh toán',
    'subscription_type': 'Loại gói (Basic, Premium, VIP)',
    'has_active_promo': 'Có sử dụng khuyến mại (Yes/No)',
    'app_rating': 'Đánh giá app (1-5 sao)',
    'num_notification_opt_out': 'Số lần tắt thông báo',
    'customer_support_satisfaction': 'Mức độ hài lòng hỗ trợ (1-5)',
    'monthly_charges': 'Chi phí hàng tháng ($)',
    'churn_target': 'TARGET: Khách hàng rời bỏ (0/1) - Classification'
}

print("\n📋 15 Features & Target Variable:")
for i, (feature, description) in enumerate(data_structure.items(), 1):
    print(f"  {i:2d}. {feature:30s} → {description}")


📊 THIẾT KẾ CẤU TRÚC DỮ LIỆU KHÁCH HÀNG CHURN

📋 15 Features & Target Variable:
   1. customer_id                    → ID khách hàng duy nhất
   2. age                            → Tuổi khách hàng (18-70)
   3. account_tenure_months          → Thời gian sử dụng app (tháng)
   4. monthly_active_days            → Số ngày hoạt động/tháng
   5. avg_daily_usage_minutes        → Trung bình thời gian sử dụng/ngày
   6. num_sessions_per_week          → Số phiên hoạt động/tuần
   7. support_tickets_30d            → Số ticket hỗ trợ 30 ngày gần nhất
   8. payment_delay_days             → Số ngày trễ thanh toán
   9. subscription_type              → Loại gói (Basic, Premium, VIP)
  10. has_active_promo               → Có sử dụng khuyến mại (Yes/No)
  11. app_rating                     → Đánh giá app (1-5 sao)
  12. num_notification_opt_out       → Số lần tắt thông báo
  13. customer_support_satisfaction  → Mức độ hài lòng hỗ trợ (1-5)
  14. monthly_charges                → Chi phí hàng tháng ($)


In [19]:
# 3. TẠO DỮ LIỆU TỔN HỢP - GENERATE SYNTHETIC DATA
print("\n" + "="*60)
print("🔄 BẮT ĐẦU SINH DỮ LIỆU...")
print("="*60)

# 3.1 Tạo ID khách hàng
customer_id = np.arange(1, n_samples + 1)

# 3.2 Tạo các biến cơ bản (realistic distributions & controlled correlation)
# -- Biến Tuổi --
age = np.random.normal(loc=40, scale=15, size=n_samples).astype(int)
age = age + np.random.normal(0, 2, size=n_samples).astype(int)  # Nhiễu nhẹ
age = np.clip(age, 18, 70)

# -- Biến Kỳ hạn --
account_tenure_months = np.random.exponential(scale=30, size=n_samples).astype(int)
account_tenure_months = account_tenure_months + np.random.normal(0, 3, size=n_samples).astype(int)  # Nhiễu nhẹ
account_tenure_months = np.clip(account_tenure_months, 1, 120)

# -- Biến GỐC: Số ngày hoạt động trong tháng --
monthly_active_days = np.random.beta(a=5, b=2, size=n_samples) * 30
outlier_mask_1 = np.random.random(n_samples) < 0.10  # 10% khách hàng đột ngột bỏ bê app
monthly_active_days[outlier_mask_1] = np.random.uniform(0.5, 5, outlier_mask_1.sum())
monthly_active_days = monthly_active_days + np.random.normal(0, 0.5, size=n_samples) # Sửa nhiễu tinh tế
monthly_active_days = np.clip(monthly_active_days, 1, 30).astype(int)

# -- Biến PHỤ THUỘC 1: Số phút sử dụng hàng ngày (Tương quan trực tiếp với số ngày active) --
# Logic: Vào app nhiều ngày xu hướng dùng nhiều phút hơn
base_minutes = (monthly_active_days / 30) * 90  # Mốc cơ sở tối đa 90 phút theo tỷ lệ ngày active
usage_noise = np.random.normal(loc=0, scale=8, size=n_samples)  # Thêm biến động cá nhân vừa phải
avg_daily_usage_minutes = base_minutes + usage_noise
# Thêm đuôi phân phối dài cho nhóm "nghiện app" (Heavy users)
heavy_user_mask = np.random.random(n_samples) < 0.05
avg_daily_usage_minutes[heavy_user_mask] += np.random.exponential(scale=60, size=heavy_user_mask.sum())
avg_daily_usage_minutes = np.clip(avg_daily_usage_minutes, 5, 480)

# -- Biến PHỤ THUỘC 2: Số phiên/tuần (Tương quan trực tiếp với số ngày active) --
base_sessions = (monthly_active_days / 30) * 14  # Mốc cơ sở tối đa 14 phiên/tuần
session_noise = np.random.normal(loc=0, scale=1, size=n_samples)
num_sessions_per_week = base_sessions + session_noise
num_sessions_per_week = np.clip(num_sessions_per_week, 1, 42).astype(int)

# -- Biến Số ticket hỗ trợ --
support_tickets_30d = np.random.poisson(lam=0.5, size=n_samples).astype(int)
outlier_mask_2 = np.random.random(n_samples) < 0.05  # 5% dính lỗi hệ thống, report liên tục
support_tickets_30d[outlier_mask_2] = np.random.randint(5, 10, outlier_mask_2.sum())
support_tickets_30d = np.clip(support_tickets_30d, 0, 10)

# -- Biến Số ngày trễ thanh toán --
payment_delay_days = np.random.exponential(scale=2, size=n_samples)
outlier_mask_3 = np.random.random(n_samples) < 0.08  # 8% chây ì thanh toán
payment_delay_days[outlier_mask_3] = np.random.uniform(15, 30, outlier_mask_3.sum())
payment_delay_days = payment_delay_days + np.random.normal(0, 0.5, size=n_samples) # Sửa nhiễu nhẹ
payment_delay_days = np.clip(payment_delay_days, 0, 30).astype(int)

# 3.3 Tạo biến danh mục
subscription_type = np.random.choice(['Basic', 'Premium', 'VIP'], 
                                     size=n_samples, 
                                     p=[0.5, 0.35, 0.15])

has_active_promo = np.random.choice(['Yes', 'No'], 
                                    size=n_samples, 
                                    p=[0.4, 0.6])

# 3.4 Tạo các biến đánh giá
app_rating = np.random.choice([1, 2, 3, 4, 5], 
                              size=n_samples, 
                              p=[0.05, 0.1, 0.15, 0.3, 0.4])
random_rating_mask = np.random.random(n_samples) < 0.05
app_rating[random_rating_mask] = np.random.randint(1, 6, random_rating_mask.sum()) # Sửa từ (1,5) thành (1,6) để lấy được điểm 5

num_notification_opt_out = np.random.poisson(lam=1.5, size=n_samples).astype(int)
num_notification_opt_out = num_notification_opt_out + np.random.normal(0, 0.5, size=n_samples).astype(int)
num_notification_opt_out = np.clip(num_notification_opt_out, 0, 20)

customer_support_satisfaction = np.random.choice([1, 2, 3, 4, 5], 
                                                 size=n_samples, 
                                                 p=[0.08, 0.12, 0.15, 0.25, 0.4])

# 3.5 Tạo chi phí hàng tháng (theo loại gói + outliers)
monthly_charges = np.zeros(n_samples)
for i in range(n_samples):
    if subscription_type[i] == 'Basic':
        monthly_charges[i] = np.random.normal(loc=9.99, scale=1.5, size=1)[0]
    elif subscription_type[i] == 'Premium':
        monthly_charges[i] = np.random.normal(loc=19.99, scale=2.5, size=1)[0]
    else:  # VIP
        monthly_charges[i] = np.random.normal(loc=49.99, scale=5, size=1)[0]

charge_outlier_mask = np.random.random(n_samples) < 0.03
monthly_charges[charge_outlier_mask] = np.random.uniform(5, 100, charge_outlier_mask.sum())
monthly_charges = np.clip(monthly_charges, 5, 100)

print(f"✅ Đã tạo {n_samples:,} records với các biến độc lập")




🔄 BẮT ĐẦU SINH DỮ LIỆU...
✅ Đã tạo 50,000 records với các biến độc lập


In [20]:
# 4. TẠO BIẾN MỤC TIÊU VỚI MỐI QUAN HỆ LOGIC - TARGET VARIABLE WITH LOGICAL RELATIONSHIPS
print("\n" + "="*60)
print("🔧 TÍNH TOÁN CHURN VỚI TỈ LỆ HỢP LÝ (10 LOGIC)")
print("="*60)
np.random.seed(42)
# Tính toán lại churn probability với trọng số cân bằng
churn_probability_new = np.zeros(n_samples)

# 1. Trọng số Kỳ hạn (Tenure)
tenure_factor = np.where(account_tenure_months < 6, 0.20, 
                         np.where(account_tenure_months < 12, 0.10, 
                         np.where(account_tenure_months < 24, 0.02, -0.05)))

# 2. Trọng số Tương tác (Engagement) - Trục xương sống gánh chỉ số AUC-ROC
engagement_factor = (monthly_active_days / 30) * (avg_daily_usage_minutes / 100) * (num_sessions_per_week / 10)
engagement_factor = np.clip(engagement_factor, 0, 1)
engagement_factor = 0.30 - (engagement_factor * 0.50)  # High engagement = Low churn

# 3. Trọng số Khiếu nại (Support)
support_factor = np.clip((support_tickets_30d / 10) * 0.15, 0, 0.15)

# 4. Trọng số Trễ thanh toán (Payment Delay)
payment_factor = np.clip((payment_delay_days / 30) * 0.20, 0, 0.20)

# 5. Trọng số Tắt thông báo (Notification Opt-out)
notification_factor = np.clip((num_notification_opt_out / 15) * 0.05, 0, 0.05)

# 6. Trọng số Đánh giá App (Rating)
rating_factor = (6 - app_rating) / 5 * 0.08  

# 7. Trọng số Hài lòng hỗ trợ (Support Satisfaction)
support_satisfaction_factor = (6 - customer_support_satisfaction) / 5 * 0.06

# 8. Trọng số Loại gói (Subscription Type)
subscription_factor = np.where(subscription_type == 'Basic', 0.04,
                               np.where(subscription_type == 'Premium', 0.01, -0.02))

# 9. Trọng số Khuyến mãi (Promo)
promo_factor = np.where(has_active_promo == 'Yes', -0.04, 0.03)

# 10. Trọng số Nhạy cảm giá (Price Sensitivity)
price_sensitivity_factor = np.where(monthly_charges > 40, -0.02,
                                    np.where(monthly_charges > 20, 0.00, 0.02))

# --- TÍNH TỔNG HỢP LOGIC NỀN TẢNG ---
base_prob = (tenure_factor + engagement_factor + support_factor + 
             payment_factor + notification_factor + rating_factor + 
             support_satisfaction_factor + subscription_factor + promo_factor +
             price_sensitivity_factor)

# --- 🎯 GIẢI PHÁP ÉP TỶ LỆ CHURN CỐ ĐỊNH THEO PHÂN VỊ ---
# Tìm điểm ngưỡng (threshold) toán học sao cho đúng 22% khách hàng tệ nhất (base_prob cao nhất) bị đánh nhãn Churn
threshold = np.percentile(base_prob, 100 - 22) 
churn_target_new = (base_prob >= threshold).astype(int)

# Tiêm đúng 4% nhiễu ngẫu nhiên (Label Flipping) để tạo ca ngoại lệ thực tế cho mô hình học
# Giữ mức 4% đảm bảo ranh giới siêu mịn cho AUC đạt 0.8x và Recall đạt 0.8x
noise_mask = np.random.random(n_samples) < 0.04
churn_target_new = np.where(noise_mask, 1 - churn_target_new, churn_target_new)


churn_rate_new = churn_target_new.mean() * 100
print(f"\n✅ Churn rate mới: {churn_rate_new:.2f}% (Target: 15-30%)")
print(f"✨ Sử dụng 10 LOGIC trong công thức tính churn probability")

# # Cập nhật DataFrame

print(f"\n📊 Churn Distribution (Mới):")
print(f"   No Churn (0): {(churn_target_new == 0).sum():,} ({(churn_target_new == 0).mean() * 100:.2f}%)")
print(f"   Churn (1):    {(churn_target_new == 1).sum():,} ({(churn_target_new == 1).mean() * 100:.2f}%)")


🔧 TÍNH TOÁN CHURN VỚI TỈ LỆ HỢP LÝ (10 LOGIC)

✅ Churn rate mới: 24.24% (Target: 15-30%)
✨ Sử dụng 10 LOGIC trong công thức tính churn probability

📊 Churn Distribution (Mới):
   No Churn (0): 37,881 (75.76%)
   Churn (1):    12,119 (24.24%)


In [21]:
# 5. FEATURE ENGINEERING - TẠO CÁC BIẾN THỜI GIAN (3M, 6M, 12M)
print("\n" + "="*60)
print("⚙️ FEATURE ENGINEERING - TIME WINDOWS (3M, 6M, 12M)")
print("="*60)

# Tạo các biến rolling aggregate dựa trên thời gian tenure
# Logic: Khách hàng có tenure < 3 tháng → recent feature
# Khách hàng có tenure < 6 tháng → 6m feature
# Khách hàng có tenure < 12 tháng → 12m feature

# 5.1 Feature aggregates cho 3 tháng gần nhất (3M)
avg_active_days_3m = np.where(account_tenure_months >= 3,
                               monthly_active_days + np.random.normal(0, 2, n_samples),
                               monthly_active_days * (account_tenure_months / 3))
avg_active_days_3m = np.clip(avg_active_days_3m, 0, 30)

usage_minutes_3m = np.where(account_tenure_months >= 3,
                             avg_daily_usage_minutes + np.random.normal(0, 10, n_samples),
                             avg_daily_usage_minutes * (account_tenure_months / 3))
usage_minutes_3m = np.clip(usage_minutes_3m, 0, 500)

support_tickets_3m = np.where(account_tenure_months >= 3,
                               support_tickets_30d * 3,
                               support_tickets_30d * (account_tenure_months / 3))
support_tickets_3m = support_tickets_3m.astype(int)

# 5.2 Feature aggregates cho 6 tháng (6M)
avg_active_days_6m = np.where(account_tenure_months >= 6,
                               monthly_active_days + np.random.normal(0, 2, n_samples),
                               monthly_active_days * min(account_tenure_months.max() / 6, 1) 
                                 if account_tenure_months.max() > 0 else monthly_active_days)
avg_active_days_6m = np.clip(avg_active_days_6m, 0, 30)

usage_minutes_6m = np.where(account_tenure_months >= 6,
                             avg_daily_usage_minutes + np.random.normal(0, 15, n_samples),
                             avg_daily_usage_minutes)
usage_minutes_6m = np.clip(usage_minutes_6m, 0, 500)

support_tickets_6m = np.where(account_tenure_months >= 6,
                               support_tickets_30d * 6,
                               support_tickets_30d * (account_tenure_months / 6))
support_tickets_6m = support_tickets_6m.astype(int)

# 5.3 Feature aggregates cho 12 tháng (12M)
avg_active_days_12m = np.where(account_tenure_months >= 12,
                                monthly_active_days + np.random.normal(0, 2, n_samples),
                                monthly_active_days)
avg_active_days_12m = np.clip(avg_active_days_12m, 0, 30)

usage_minutes_12m = np.where(account_tenure_months >= 12,
                              avg_daily_usage_minutes + np.random.normal(0, 20, n_samples),
                              avg_daily_usage_minutes)
usage_minutes_12m = np.clip(usage_minutes_12m, 0, 500)

support_tickets_12m = np.where(account_tenure_months >= 12,
                                support_tickets_30d * 12,
                                support_tickets_30d * (account_tenure_months / 12))
support_tickets_12m = support_tickets_12m.astype(int)

print(f"✅ Tạo 9 biến engineering cho các time windows (3M, 6M, 12M)")


⚙️ FEATURE ENGINEERING - TIME WINDOWS (3M, 6M, 12M)
✅ Tạo 9 biến engineering cho các time windows (3M, 6M, 12M)


In [22]:
# 6. TẠO DATAFRAME - CREATE DATAFRAME
print("\n" + "="*60)
print("📦 TẠO DATAFRAME VÀ KIỂM TRA DỮ LIỆU")
print("="*60)

df = pd.DataFrame({
    # Thông tin cơ bản
    'customer_id': customer_id,
    'age': age,
    'account_tenure_months': account_tenure_months,
    'monthly_active_days': monthly_active_days,
    'avg_daily_usage_minutes': avg_daily_usage_minutes,
    'num_sessions_per_week': num_sessions_per_week,
    'support_tickets_30d': support_tickets_30d,
    'payment_delay_days': payment_delay_days,
    'subscription_type': subscription_type,
    'has_active_promo': has_active_promo,
    'app_rating': app_rating,
    'num_notification_opt_out': num_notification_opt_out,
    'customer_support_satisfaction': customer_support_satisfaction,
    'monthly_charges': monthly_charges,
    
    # Biến target
    'churn_target': churn_target_new
})

# Thêm các biến feature engineering (time windows)
df['avg_active_days_3m'] = avg_active_days_3m
df['usage_minutes_3m'] = usage_minutes_3m
df['support_tickets_3m'] = support_tickets_3m
df['avg_active_days_6m'] = avg_active_days_6m
df['usage_minutes_6m'] = usage_minutes_6m
df['support_tickets_6m'] = support_tickets_6m
df['avg_active_days_12m'] = avg_active_days_12m
df['usage_minutes_12m'] = usage_minutes_12m
df['support_tickets_12m'] = support_tickets_12m

print(f"\n✅ Dataframe được tạo thành công!")
print(f"   📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   📋 Columns: {list(df.columns)}")
print(f"\n📍 Thông tin Dataset:")
print(df.head(10))


📦 TẠO DATAFRAME VÀ KIỂM TRA DỮ LIỆU

✅ Dataframe được tạo thành công!
   📊 Shape: 50,000 rows × 24 columns
   📋 Columns: ['customer_id', 'age', 'account_tenure_months', 'monthly_active_days', 'avg_daily_usage_minutes', 'num_sessions_per_week', 'support_tickets_30d', 'payment_delay_days', 'subscription_type', 'has_active_promo', 'app_rating', 'num_notification_opt_out', 'customer_support_satisfaction', 'monthly_charges', 'churn_target', 'avg_active_days_3m', 'usage_minutes_3m', 'support_tickets_3m', 'avg_active_days_6m', 'usage_minutes_6m', 'support_tickets_6m', 'avg_active_days_12m', 'usage_minutes_12m', 'support_tickets_12m']

📍 Thông tin Dataset:
   customer_id  age  account_tenure_months  monthly_active_days  \
0            1   47                     13                   20   
1            2   37                     34                    1   
2            3   50                      5                   26   
3            4   65                     46                   23   
4      

In [23]:
# 7. KIỂM TRA TÍNH HỢP LÝ CỦA DỮ LIỆU - DATA VALIDATION
print("\n" + "="*60)
print("✔️ KIỂM TRA TÍNH HỢP LÝ VÀ THỰC TẾ CỦA DỮ LIỆU")
print("="*60)

print("\n🔍 Thống kê mô tả (Descriptive Statistics):")
print(df.describe().round(2))

print("\n" + "-"*60)
print("📊 Kiểm tra Missing Values:")
print(df.isnull().sum())

print("\n" + "-"*60)
print("🎯 Phân bố Target Variable (Churn):")
churn_counts = df['churn_target'].value_counts()
churn_pct = df['churn_target'].value_counts(normalize=True) * 100
print(f"   ❌ Churn (1):     {churn_counts[1]:,} ({churn_pct[1]:.2f}%)")
print(f"   ✅ No Churn (0):  {churn_counts[0]:,} ({churn_pct[0]:.2f}%)")

print("\n" + "-"*60)
print("📌 Kiểm tra Logic - Mối Quan Hệ Giữa Biến:")

# Biến 1: Khách hàng mới (tenure < 6 tháng) có churn (bỏ app) cao hơn
new_cust_churn = df[df['account_tenure_months'] < 6]['churn_target'].mean() * 100
loyal_cust_churn = df[df['account_tenure_months'] >= 24]['churn_target'].mean() * 100
print(f"   ✓ Khách hàng mới (tenure < 6m): {new_cust_churn:.2f}% churn")
print(f"   ✓ Khách hàng trung thành (tenure >= 24m): {loyal_cust_churn:.2f}% churn")
print(f"   → Kỳ vọng: Khách hàng mới churn nhiều hơn ✅" if new_cust_churn > loyal_cust_churn else "   → Cảnh báo: Logic bất thường")

# Biến 2: Khách hàng sử dụng ít (active_days < 10) có churn cao
low_engagement_churn = df[df['monthly_active_days'] < 10]['churn_target'].mean() * 100
high_engagement_churn = df[df['monthly_active_days'] > 20]['churn_target'].mean() * 100
print(f"\n   ✓ Engagement thấp (active_days < 10): {low_engagement_churn:.2f}% churn")
print(f"   ✓ Engagement cao (active_days > 20): {high_engagement_churn:.2f}% churn")
print(f"   → Kỳ vọng: Engagement thấp churn nhiều hơn ✅" if low_engagement_churn > high_engagement_churn else "   → Cảnh báo: Logic bất thường")

# Biến 3: Gói Basic có churn cao hơn Premium
basic_churn = df[df['subscription_type'] == 'Basic']['churn_target'].mean() * 100
premium_churn = df[df['subscription_type'] == 'Premium']['churn_target'].mean() * 100
vip_churn = df[df['subscription_type'] == 'VIP']['churn_target'].mean() * 100
print(f"\n   ✓ Basic package: {basic_churn:.2f}% churn")
print(f"   ✓ Premium package: {premium_churn:.2f}% churn")
print(f"   ✓ VIP package: {vip_churn:.2f}% churn")
print(f"   → Kỳ vọng: Basic > Premium > VIP ✅" if (basic_churn > premium_churn > vip_churn) else "   → Cảnh báo: Logic bất thường")

# Biến 4: Nhiều support tickets → churn cao
many_tickets_churn = df[df['support_tickets_30d'] >= 2]['churn_target'].mean() * 100
few_tickets_churn = df[df['support_tickets_30d'] == 0]['churn_target'].mean() * 100
print(f"\n   ✓ Có nhiều support tickets (>=2): {many_tickets_churn:.2f}% churn")
print(f"   ✓ Không có support tickets (0): {few_tickets_churn:.2f}% churn")
print(f"   → Kỳ vọng: Nhiều tickets → churn cao hơn ✅" if many_tickets_churn > few_tickets_churn else "   → Cảnh báo: Logic bất thường")

# Biến 5: Đánh giá app thấp → churn cao
low_rating_churn = df[df['app_rating'] <= 2]['churn_target'].mean() * 100
high_rating_churn = df[df['app_rating'] >= 4]['churn_target'].mean() * 100
print(f"\n   ✓ Đánh giá app thấp (<=2 sao): {low_rating_churn:.2f}% churn")
print(f"   ✓ Đánh giá app cao (>=4 sao): {high_rating_churn:.2f}% churn")
print(f"   → Kỳ vọng: Rating thấp → churn cao ✅" if low_rating_churn > high_rating_churn else "   → Cảnh báo: Logic bất thường")

print("\n" + "="*60)
print("✅ DỮ LIỆU ĐÃ ĐƯỢC KIỂM TRA VÀ HỢP LÝ!")


✔️ KIỂM TRA TÍNH HỢP LÝ VÀ THỰC TẾ CỦA DỮ LIỆU

🔍 Thống kê mô tả (Descriptive Statistics):
       customer_id       age  account_tenure_months  monthly_active_days  \
count     50000.00  50000.00                50000.0              50000.0   
mean      25000.50     39.90                   29.3                 19.1   
std       14433.90     13.79                   27.9                  7.2   
min           1.00     18.00                    1.0                  1.0   
25%       12500.75     29.00                    8.0                 16.0   
50%       25000.50     40.00                   21.0                 21.0   
75%       37500.25     50.00                   41.0                 24.0   
max       50000.00     70.00                  120.0                 30.0   

       avg_daily_usage_minutes  num_sessions_per_week  support_tickets_30d  \
count                 50000.00               50000.00             50000.00   
mean                     60.39                   8.48              

In [24]:
# 8. LƯU DỮ LIỆU - SAVE DATA TO CSV
print("\n" + "="*60)
print("💾 LƯU DỮ LIỆU TỚI FILE CSV")
print("="*60)

# Tạo thư mục nếu chưa tồn tại
os.makedirs('Data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Lưu file
output_path = r'/Users/thanhnhat/Documents/mock project/Data/churn_data_customer.csv'
df.to_csv(output_path, index=False)

print(f"\n✅ File đã được lưu thành công!")
print(f"   📁 Đường dẫn: {os.path.abspath(output_path)}")
print(f"   📊 Kích thước: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   💾 File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

# Hiển thị thông tin cuối cùng
print("\n" + "="*60)
print("📋 TỔNG HỢP DATASET:")
print("="*60)
print(f"\n📊 Dataset Info:")
print(df.info())

print(f"\n📈 Target Distribution:")
print(df['churn_target'].value_counts().sort_index())

print(f"\n🎯 Subscription Type Distribution:")
print(df['subscription_type'].value_counts())

print(f"\n✨ Dataset đã sẵn sàng cho việc xây dựng mô hình ML!")
print(f"   Bạn có thể sử dụng file: {output_path}")
print(f"   Để bắt đầu training mô hình dự đoán churn.")


💾 LƯU DỮ LIỆU TỚI FILE CSV

✅ File đã được lưu thành công!
   📁 Đường dẫn: /Users/thanhnhat/Documents/mock project/Data/churn_data_customer.csv
   📊 Kích thước: 50,000 rows × 24 columns
   💾 File size: 8.66 MB

📋 TỔNG HỢP DATASET:

📊 Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   customer_id                    50000 non-null  int64  
 1   age                            50000 non-null  int64  
 2   account_tenure_months          50000 non-null  int64  
 3   monthly_active_days            50000 non-null  int64  
 4   avg_daily_usage_minutes        50000 non-null  float64
 5   num_sessions_per_week          50000 non-null  int64  
 6   support_tickets_30d            50000 non-null  int64  
 7   payment_delay_days             50000 non-null  int64  
 8   subscription_type              50000 non-null  s

In [25]:
# 9. KIỂM TRA ĐƯỜNG DẪN VÀ KIỂM FIX CHỈ SỐ CHURN
import os

print("Current Working Directory:", os.getcwd())
print("Files in current directory:", os.listdir('.'))

# Kiểm tra xem file đã lưu hay chưa
if os.path.exists('Data/customer_churn_dataset_generated.csv'):
    print("✅ File đã tìm thấy!")
    print(f"File size: {os.path.getsize('Data/customer_churn_dataset_generated.csv') / 1024 / 1024:.2f} MB")
else:
    print("❌ File không tìm thấy, đang lưu lại...")
    df.to_csv('Data/customer_churn_dataset_generated.csv', index=False)
    print("✅ File đã được lưu thành công!")

Current Working Directory: /Users/thanhnhat/Documents/mock project
Files in current directory: ['.DS_Store', 'models', '1_Data_Generation.ipynb', 'Mini Project ML Requirements - Yeu cau de bai.md', '2_Exploratory_Data_Analysis.ipynb', 'Data', '3_Model_Building_moi.ipynb']
❌ File không tìm thấy, đang lưu lại...
✅ File đã được lưu thành công!


In [26]:
# 10. KIỂM TRA CUỐI CÙNG - FINAL COMPREHENSIVE VALIDATION
print("\n" + "="*70)
print("✅ KIỂM TRA CUỐI CÙNG VÀ THỐNG KÊ TOÀN BỘ DATASET")
print("="*70)

print("\n📊 THÔNG TIN CHUNG:")
print(f"   Total Records: {df.shape[0]:,}")
print(f"   Total Features: {df.shape[1]}")
print(f"   File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

print("\n📈 MỖI BIẾN - THỐNG KÊ MÔ TẢ:")
stats = df.describe().round(2)
print(stats)

print("\n🎯 PHÂN TÍCH MỐI QUAN HỆ LOGIC GIỮA CHURN VÀ CÁC BIẾN:")

print("\n  1️⃣ Tenure (Thời gian sử dụng):")
tenure_labels = ['0-6m', '6-12m', '12-24m', '24m+']
tenure_bins = [0, 6, 12, 24, 120]
df_temp = df.copy()
df_temp['tenure_group'] = pd.cut(df_temp['account_tenure_months'], bins=tenure_bins, labels=tenure_labels)
for label in tenure_labels:
    mask = df_temp['tenure_group'] == label
    churn_pct = df[mask]['churn_target'].mean() * 100
    count = mask.sum()
    print(f"      {label:20s}: {count:6,} customers, {churn_pct:5.1f}% churn")
print(f"      ✓ Khách hàng mới (0-6m) churn cao hơn khách hàng trung thành")

print("\n  2️⃣ Engagement (Hoạt động hàng tháng):")
engagement_labels = ['0-5d', '5-10d', '10-20d', '20-30d']
engagement_bins = [0, 5, 10, 20, 30]
df_temp['engagement_group'] = pd.cut(df_temp['monthly_active_days'], bins=engagement_bins, labels=engagement_labels)
for label in engagement_labels:
    mask = df_temp['engagement_group'] == label
    churn_pct = df[mask]['churn_target'].mean() * 100
    count = mask.sum()
    print(f"      {label:20s}: {count:6,} customers, {churn_pct:5.1f}% churn")
print(f"      ✓ Engagement thấp → Churn cao")

print("\n  3️⃣ Subscription Type (Loại gói):")
for sub_type in ['Basic', 'Premium', 'VIP']:
    mask = df['subscription_type'] == sub_type
    churn_pct = df[mask]['churn_target'].mean() * 100
    count = mask.sum()
    print(f"      {sub_type:20s}: {count:6,} customers, {churn_pct:5.1f}% churn")
print(f"      ✓ Basic package có churn cao nhất (khách hàng thử)")

print("\n  4️⃣ Support Tickets (30 ngày gần nhất):")
for ticket_count in [0, 1, 2]:
    if ticket_count == 2:
        mask = df['support_tickets_30d'] >= 2
        label = f"{ticket_count}+"
    else:
        mask = df['support_tickets_30d'] == ticket_count
        label = str(ticket_count)
    churn_pct = df[mask]['churn_target'].mean() * 100
    count = mask.sum()
    print(f"      {label:20s}: {count:6,} customers, {churn_pct:5.1f}% churn")
print(f"      ✓ Nhiều support tickets → Churn cao (có vấn đề)")

print("\n  5️⃣ App Rating (Đánh giá):")
for rating in [1, 2, 3, 4, 5]:
    mask = df['app_rating'] == rating
    churn_pct = df[mask]['churn_target'].mean() * 100
    count = mask.sum()
    print(f"      {rating}-star rating: {count:6,} customers, {churn_pct:5.1f}% churn")
print(f"      ✓ Rating thấp → Churn cao")

print("\n  6️⃣ Promo Usage (Sử dụng khuyến mại):")
for promo in ['Yes', 'No']:
    mask = df['has_active_promo'] == promo
    churn_pct = df[mask]['churn_target'].mean() * 100
    count = mask.sum()
    print(f"      {promo:20s}: {count:6,} customers, {churn_pct:5.1f}% churn")
print(f"      ✓ Khách hàng sử dụng promo ít bỏ cuộc hơn")

print("\n  7️⃣ Payment Delay (Trễ thanh toán):")
delay_labels = ['0-5d', '5-10d', '10-30d']
delay_bins = [0, 5, 10, 30]
df_temp['delay_group'] = pd.cut(df_temp['payment_delay_days'], bins=delay_bins, labels=delay_labels)
for label in delay_labels:
    mask = df_temp['delay_group'] == label
    churn_pct = df[mask]['churn_target'].mean() * 100
    count = mask.sum()
    print(f"      {label:20s}: {count:6,} customers, {churn_pct:5.1f}% churn")
print(f"      ✓ Trễ thanh toán → Churn cao")

print("\n" + "="*70)
print("✨ DATASET ĐÃ HOÀN THÀNH VÀ SẴN SÀNG CHO MÔ HÌNH ML!")
print("="*70)
print(f"\n📁 Vị trí file: {os.path.abspath(output_path)}")
print(f"📊 Kích thước: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"🎯 Target variable: 'churn_target' (Classification - 0/1)")
print(f"💾 File format: CSV")
print(f"\n🚀 Sẵn sàng cho các bước tiếp theo:")
print(f"   • Exploratory Data Analysis (EDA)")
print(f"   • Feature Selection")
print(f"   • Model Training (Logistic Regression, Random Forest, XGBoost, ...)")
print(f"   • Model Evaluation")


✅ KIỂM TRA CUỐI CÙNG VÀ THỐNG KÊ TOÀN BỘ DATASET

📊 THÔNG TIN CHUNG:
   Total Records: 50,000
   Total Features: 24
   File size: 8.66 MB

📈 MỖI BIẾN - THỐNG KÊ MÔ TẢ:
       customer_id       age  account_tenure_months  monthly_active_days  \
count     50000.00  50000.00                50000.0              50000.0   
mean      25000.50     39.90                   29.3                 19.1   
std       14433.90     13.79                   27.9                  7.2   
min           1.00     18.00                    1.0                  1.0   
25%       12500.75     29.00                    8.0                 16.0   
50%       25000.50     40.00                   21.0                 21.0   
75%       37500.25     50.00                   41.0                 24.0   
max       50000.00     70.00                  120.0                 30.0   

       avg_daily_usage_minutes  num_sessions_per_week  support_tickets_30d  \
count                 50000.00               50000.00             50